<a href="https://colab.research.google.com/github/haidy47-design/Petique-Backend/blob/main/shein_scrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install selenium pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import WebDriverException


def setup_driver(headless=True):
    """Create and configure Selenium Chrome driver."""

    chrome_options = Options()

    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=chrome_options)
    return driver


def clean_product_name(name):
    """Clean product name from extra spaces and unwanted characters."""

    if not name:
        return None

    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"[^\w\s\u0600-\u06FF\-]", "", name)

    return name


def extract_currency(price_text):
    """Extract currency from raw price text."""

    if not price_text:
        return None

    price_text = price_text.upper()

    currency_patterns = ["SAR", "RS", "ر.س", "$", "USD", "EGP", "AED"]

    for currency in currency_patterns:
        if currency in price_text:
            return currency

    return None


def clean_price(price_text):
    """Remove currency symbols and convert price to float."""

    if not price_text:
        return None

    price_text = price_text.replace(",", "")
    price_number = re.sub(r"[^\d.]", "", price_text)

    try:
        return float(price_number)
    except ValueError:
        return None


def clean_image_url(image_url):
    """Ensure image URL is valid and uses HTTPS."""

    if not image_url:
        return None

    if image_url.startswith("//"):
        image_url = "https:" + image_url

    if image_url.startswith("http://"):
        image_url = image_url.replace("http://", "https://")

    # Try to get higher quality image if SHEIN uses thumbnail params
    image_url = re.sub(r"_thumbnail_\d+x\d+", "", image_url)

    return image_url


def scrape_shein_category(url, category, max_items=50, headless=True):
    """Scrape product data from SHEIN category page."""

    driver = setup_driver(headless=headless)
    products = []

    try:
        print("Opening category page...")
        driver.get(url)
        time.sleep(5)

        print("Scrolling page to load dynamic products...")
        for _ in range(6):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

        product_cards = driver.find_elements(By.CSS_SELECTOR, "section, div")

        print(f"Found {len(product_cards)} possible product elements.")

        count = 0

        for card in product_cards:
            if count >= max_items:
                break

            try:
                name = None
                price = None
                image_url = None
                product_link = None

                try:
                    name = card.find_element(By.CSS_SELECTOR, "a").get_attribute("aria-label")
                except:
                    pass

                try:
                    price = card.find_element(By.CSS_SELECTOR, "[class*='price']").text
                except:
                    pass

                try:
                    image_url = card.find_element(By.CSS_SELECTOR, "img").get_attribute("src")
                except:
                    pass

                try:
                    product_link = card.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
                except:
                    pass

                if not name or not price:
                    continue

                count += 1
                print(f"Scraping item {count} of {max_items}...")

                products.append({
                    "Product Name": name,
                    "Price Raw": price,
                    "Image URL": image_url,
                    "Product Link": product_link,
                    "Category": category
                })

            except Exception as e:
                print(f"Skipping product due to error: {e}")

    except WebDriverException as e:
        print(f"Selenium error: {e}")

    finally:
        driver.quit()
        print("Browser closed.")

    return products


def clean_data(products):
    """Clean scraped product data using Pandas."""

    df = pd.DataFrame(products)

    if df.empty:
        print("No data scraped.")
        return df

    print("Cleaning data...")

    df["Product Name"] = df["Product Name"].apply(clean_product_name)
    df["Currency"] = df["Price Raw"].apply(extract_currency)
    df["Price"] = df["Price Raw"].apply(clean_price)
    df["Image URL"] = df["Image URL"].apply(clean_image_url)

    df.dropna(subset=["Product Name", "Price"], inplace=True)

    df = df[
        [
            "Product Name",
            "Price",
            "Currency",
            "Price Raw",
            "Image URL",
            "Product Link",
            "Category"
        ]
    ]

    df.drop_duplicates(subset=["Product Name", "Product Link"], inplace=True)

    return df


def export_to_csv(df, output_file):
    """Export cleaned data to CSV using UTF-8 encoding."""

    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Data exported successfully to: {output_file}")


def main():
    category_url = "PASTE_SHEIN_CATEGORY_URL_HERE"
    category_name = "Women Fashion"
    output_file = "shein_products_cleaned.csv"

    products = scrape_shein_category(
        url=category_url,
        category=category_name,
        max_items=50,
        headless=True
    )

    cleaned_df = clean_data(products)

    if not cleaned_df.empty:
        export_to_csv(cleaned_df, output_file)
        print(cleaned_df.head())
    else:
        print("No valid data to export.")


if __name__ == "__main__":
    main()

SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x58a50029e8aa <unknown>
#1 0x58a4ffc998a9 <unknown>
#2 0x58a4ffcd7e5b <unknown>
#3 0x58a4ffcd2b29 <unknown>
#4 0x58a4ffd22316 <unknown>
#5 0x58a4ffd219fc <unknown>
#6 0x58a4ffce14cf <unknown>
#7 0x58a4ffce2291 <unknown>
#8 0x58a50026476b <unknown>
#9 0x58a50026767d <unknown>
#10 0x58a5002516c8 <unknown>
#11 0x58a500268210 <unknown>
#12 0x58a500238170 <unknown>
#13 0x58a50028b7b8 <unknown>
#14 0x58a50028b955 <unknown>
#15 0x58a50029d30e <unknown>
#16 0x78c87996dac3 <unknown>


In [ ]:
def setup_driver(headless=True):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service

    chrome_options = Options()

    if headless:
        chrome_options.add_argument("--headless=new")

    # Important for Colab / Linux servers
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--remote-debugging-port=9222")
    chrome_options.add_argument("--window-size=1920,1080")

    # Avoid automation crash/detection issues
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-popup-blocking")

    driver = webdriver.Chrome(options=chrome_options)

    return driver

In [ ]:
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium pandas

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Package

In [ ]:
def setup_driver(headless=True):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service

    chrome_options = Options()

    chrome_options.binary_location = "/usr/bin/chromium-browser"

    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--remote-debugging-port=9222")
    chrome_options.add_argument("--window-size=1920,1080")

    service = Service("/usr/bin/chromedriver")

    driver = webdriver.Chrome(
        service=service,
        options=chrome_options
    )

    return driver